# Embed the Filing and Firm Summaries

| Item | Description |
|---|---|
| Goal | Embed every filing summary and every firm summary with the encoder and pooling of the policy summary vector (T3.2.5), so that T3.4.1 can take cosines between them. All prompt versions in the tables are embedded, so the versions can be compared. |
| Input | `filing_summaries.parquet` (T3.3.3) and `firm_summaries.parquet` (T3.3.5) in the T3.3 Processed folder. |
| Output | In the T3.3 Processed folder: `filing_summary_embeddings.npy` (n × 1024, float32) with `filing_summary_embedding_keys.parquet` in the same row order; `firm_summary_embeddings.npy` with `firm_summary_embedding_keys.parquet`; `embedding_record.json`. |
| Model | `BAAI/bge-large-en-v1.5`, CLS token of the last layer, L2-normalized, inputs at most 512 tokens; every summary is within the limit by construction. |
| Run | Rerun 2026-10-31 on one A40, about six minutes; the vectors equal those of the October 2026 run. |


## Step 1. Import packages


In [1]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import AutoModel, AutoTokenizer


## Step 2. Define file locations and settings


In [2]:
PROJECT_FOLDER = Path.cwd().resolve().parents[2]  # <pipeline>/<section>/<subfolder>/ sits three levels below the clone
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis")  # the one line to change on another machine
NOTEBOOK_SUBFOLDER = Path("T - Three Step Text Analysis Pipeline/T3 - Topic Relevance and Corporate Disclosure/T3.3 - Firm Side - Summaries of the 10-K Item 1 Filings")
NOTEBOOK_NAME = "T3.3.6 - Embed - The Filing and Firm Summary Vectors"  # the notebook filename without .ipynb
RAW_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Raw"
INTERIM_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Interim"
PROCESSED_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Processed"
FILING_TABLE_FILE = PROCESSED_FOLDER / "filing_summaries.parquet"
FIRM_TABLE_FILE = PROCESSED_FOLDER / "firm_summaries.parquet"

BATCH_SIZE = 64  # summaries per forward pass
FILING_KEYS = ["accessionNo", "family", "prompt_version"]
FIRM_KEYS = ["gvkey", "set", "family", "prompt_version"]
FIGURE_FOLDER = PROJECT_FOLDER / "R - Results/figures" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
TABLE_FOLDER = PROJECT_FOLDER / "R - Results/tables" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
MODEL_CACHE = SCRATCH_FOLDER / "cache" / "huggingface"
ENV_FILE = PROJECT_FOLDER / "T - Three Step Text Analysis Pipeline/T3 - Topic Relevance and Corporate Disclosure" / ".env"
for folder in (PROCESSED_FOLDER,):
    folder.mkdir(parents=True, exist_ok=True)


## Step 3. Point the Hugging Face cache at the scratch root


In [3]:
MODEL_CACHE.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(MODEL_CACHE)
# A token is needed only for gated models. Qwen3.5-4B and bge-large-en-v1.5 are open; the lines stay so the cell also serves a gated model.
if ENV_FILE.exists():
    for line in ENV_FILE.read_text().splitlines():
        name, _, value = line.strip().partition("=")
        if name == "HF_TOKEN" and value:
            os.environ["HF_TOKEN"] = value.strip().strip('"').strip("'")


## Step 4. Load the two summary tables

Every row of each table gets a vector; the key columns identify the row.


In [4]:
filing_table = pd.read_parquet(FILING_TABLE_FILE)
firm_table = pd.read_parquet(FIRM_TABLE_FILE)
assert not filing_table.duplicated(FILING_KEYS).any() and not firm_table.duplicated(FIRM_KEYS).any()
print(f"Filing summaries: {len(filing_table):,} rows\n" + filing_table.groupby(["set", "family", "prompt_version"]).size().rename("rows").to_string())
print(f"Firm summaries: {len(firm_table):,} rows\n" + firm_table.groupby(["set", "family", "prompt_version"]).size().rename("rows").to_string())


Filing summaries: 22,923 rows
set          family                     prompt_version     
remaining    summary_lines_of_business  item1-mapreduce-1.3    16707
three_years  summary_lines_of_business  item1-mapreduce-1.1     2072
                                        item1-mapreduce-1.2     2072
                                        item1-mapreduce-1.3     2072
Firm summaries: 1,899 rows
set          family                     prompt_version     
all_years    summary_lines_of_business  item1-mapreduce-1.3    1058
three_years  summary_lines_of_business  item1-mapreduce-1.3     841


## Step 5. Load the encoder


In [5]:
ENCODER = "BAAI/bge-large-en-v1.5"  # the encoder of T1.3.2 and T2; CLS pooling is the pooling this checkpoint publishes
MAX_TOKENS = 512  # the encoder's input limit

tokenizer = AutoTokenizer.from_pretrained(ENCODER)
assert torch.cuda.is_available(), "run this notebook on a GPU node"
model = AutoModel.from_pretrained(ENCODER).to("cuda").eval()
assert model.config.max_position_embeddings == MAX_TOKENS
print(f"Encoder: {ENCODER}; hidden size {model.config.hidden_size}; input limit {MAX_TOKENS} tokens")
print(f"Device: {torch.cuda.get_device_name(0)}; dtype {next(model.parameters()).dtype}")
assert filing_table["n_bge_tokens"].le(MAX_TOKENS).all() and firm_table["n_bge_tokens"].le(MAX_TOKENS).all()


Encoder: BAAI/bge-large-en-v1.5; hidden size 1024; input limit 512 tokens
Device: NVIDIA A40; dtype torch.float32


## Step 6. Embed the texts

Texts are sorted by length so each batch pads little, embedded, and put back in table order.


In [6]:
def embed_texts(texts, batch_size=64):
    """Embed texts as the CLS token of the last layer at unit length, in float32; texts are sorted by length so that each batch pads little."""
    texts = list(texts)
    order = np.argsort([len(text) for text in texts])
    vectors_sorted = np.empty((len(texts), model.config.hidden_size), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            batch = [texts[i] for i in order[start:start + batch_size]]
            encoded = tokenizer(batch, padding=True, truncation=True, max_length=MAX_TOKENS, return_tensors="pt").to("cuda")
            last_layer = model(**encoded).last_hidden_state
            vectors_sorted[start:start + len(batch)] = torch.nn.functional.normalize(last_layer[:, 0], dim=-1).cpu().numpy()
    vectors = np.empty_like(vectors_sorted)
    vectors[order] = vectors_sorted
    return vectors


started = datetime.now(timezone.utc)
filing_vectors = embed_texts(filing_table["summary"], BATCH_SIZE)
firm_vectors = embed_texts(firm_table["summary"], BATCH_SIZE)
seconds = (datetime.now(timezone.utc) - started).total_seconds()
assert filing_vectors.shape == (len(filing_table), model.config.hidden_size) and firm_vectors.shape == (len(firm_table), model.config.hidden_size)
assert np.allclose(np.linalg.norm(filing_vectors, axis=1), 1, atol=1e-4) and np.allclose(np.linalg.norm(firm_vectors, axis=1), 1, atol=1e-4)
print(f"Embedded {len(filing_vectors):,} filing summaries and {len(firm_vectors):,} firm summaries in {seconds / 60:.1f} minutes")


Embedded 22,923 filing summaries and 1,899 firm summaries in 5.8 minutes


## Step 7. Check the vectors

Five summaries re-embedded one at a time must match their batched vectors. Two sanity cosines follow: a yearly summary against its firm summary, which should be high, and the same filing's summary across prompt versions, which should be high but below one.


In [7]:
rng = np.random.default_rng(0)
check_rows = rng.choice(len(filing_table), size=5, replace=False)
single = np.vstack([embed_texts([filing_table["summary"].iloc[i]]) for i in check_rows])
print(f"Batched vs single-text cosine on 5 filing summaries: min {np.min(np.sum(single * filing_vectors[check_rows], axis=1)):.5f}")
firm_index = {tuple(key): i for i, key in enumerate(firm_table[FIRM_KEYS].itertuples(index=False, name=None))}
own = pd.Series([float(filing_vectors[i] @ firm_vectors[j]) for i, row in enumerate(filing_table.itertuples(index=False))
                 if (j := firm_index.get((row.gvkey, row.set, row.family, row.prompt_version))) is not None])
print(f"Cosine of a yearly summary with its firm summary ({len(own):,} pairs, same set and version): median {own.median():.3f}, 5th percentile {own.quantile(0.05):.3f}")
versions = sorted(filing_table["prompt_version"].unique())
by_version = {v: pd.Series(filing_table.index[filing_table["prompt_version"].eq(v)], index=filing_table.loc[filing_table["prompt_version"].eq(v), "accessionNo"]) for v in versions}
for a in range(len(versions)):
    for b in range(a + 1, len(versions)):
        common = by_version[versions[a]].index.intersection(by_version[versions[b]].index)
        cos = np.sum(filing_vectors[by_version[versions[a]][common]] * filing_vectors[by_version[versions[b]][common]], axis=1)
        print(f"Same filing, {versions[a]} vs {versions[b]} ({len(common):,} filings): median cosine {np.median(cos):.3f}")


Batched vs single-text cosine on 5 filing summaries: min 1.00000
Cosine of a yearly summary with its firm summary (2,072 pairs, same set and version): median 0.937, 5th percentile 0.873
Same filing, item1-mapreduce-1.1 vs item1-mapreduce-1.2 (2,072 filings): median cosine 0.816
Same filing, item1-mapreduce-1.1 vs item1-mapreduce-1.3 (2,072 filings): median cosine 0.811
Same filing, item1-mapreduce-1.2 vs item1-mapreduce-1.3 (2,072 filings): median cosine 0.960


## Step 8. Save the vectors, their keys, and the record


In [8]:
np.save(PROCESSED_FOLDER / "filing_summary_embeddings.npy", filing_vectors)
filing_table[FILING_KEYS].to_parquet(PROCESSED_FOLDER / "filing_summary_embedding_keys.parquet", index=False)
np.save(PROCESSED_FOLDER / "firm_summary_embeddings.npy", firm_vectors)
firm_table[FIRM_KEYS].to_parquet(PROCESSED_FOLDER / "firm_summary_embedding_keys.parquet", index=False)
record = {"model": ENCODER, "pooling": "cls", "normalized": True, "max_tokens": MAX_TOKENS, "batch_size": BATCH_SIZE, "saved_dtype": "float32",
          "n_filing_summaries": int(len(filing_table)), "n_firm_summaries": int(len(firm_table)),
          "filing_versions": {f"{s}|{f}|{v}": int(n) for (s, f, v), n in filing_table.groupby(["set", "family", "prompt_version"]).size().items()},
          "gpu": torch.cuda.get_device_name(0), "seconds": round(seconds, 1), "finished": datetime.now(timezone.utc).isoformat(timespec="seconds")}
(PROCESSED_FOLDER / "embedding_record.json").write_text(json.dumps(record, indent=2), encoding="utf-8")
print(f"Saved filing_summary_embeddings.npy {filing_vectors.shape}, firm_summary_embeddings.npy {firm_vectors.shape}, their keys, and embedding_record.json")


Saved filing_summary_embeddings.npy (22923, 1024), firm_summary_embeddings.npy (1899, 1024), their keys, and embedding_record.json
